In [ ]:
# Install psycopg2 if needed
import subprocess
subprocess.run(['pip', 'install', 'psycopg2-binary', 'sqlalchemy'], 
               capture_output=True)

import psycopg2
import pandas as pd
from psycopg2 import sql

print("✅ Libraries loaded!")

In [ ]:
# Connect to PostgreSQL
# Make sure PostgreSQL is running!
try:
    conn = psycopg2.connect(
        host='localhost',
        port=5432,
        database='postgres',  # connect to default db first
        user='postgres',
        password='postgres123'
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("✅ Connected to PostgreSQL!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
# Create bank_reviews database
try:
    cursor.execute("DROP DATABASE IF EXISTS bank_reviews;")
    cursor.execute("CREATE DATABASE bank_reviews;")
    print("✅ Database 'bank_reviews' created!")
except Exception as e:
    print(f"Error: {e}")

# Close and reconnect to new database
conn.close()

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='bank_reviews',
    user='postgres',
    password='postgres123'
)
conn.autocommit = True
cursor = conn.cursor()
print("✅ Connected to bank_reviews database!")

In [ ]:
# Create banks table
cursor.execute("""
    DROP TABLE IF EXISTS reviews CASCADE;
    DROP TABLE IF EXISTS banks CASCADE;
""")

cursor.execute("""
    CREATE TABLE banks (
        bank_id SERIAL PRIMARY KEY,
        bank_name VARCHAR(100) NOT NULL,
        app_name VARCHAR(100) NOT NULL,
        play_store_id VARCHAR(100),
        avg_rating FLOAT
    );
""")

# Create reviews table
cursor.execute("""
    CREATE TABLE reviews (
        review_id SERIAL PRIMARY KEY,
        bank_id INTEGER REFERENCES banks(bank_id),
        review_text TEXT,
        rating INTEGER CHECK (rating BETWEEN 1 AND 5),
        review_date DATE,
        sentiment_label VARCHAR(20),
        sentiment_score FLOAT,
        identified_theme VARCHAR(100),
        source VARCHAR(50) DEFAULT 'Google Play'
    );
""")

print("✅ Tables created successfully!")
print("  - banks table")
print("  - reviews table")

In [ ]:
# Insert bank metadata
banks_data = [
    ('Commercial Bank of Ethiopia', 'CBE Mobile Banking', 
     'com.combanketh.mobilebanking', 4.3),
    ('Bank of Abyssinia', 'BoA Mobile Banking', 
     'com.boa.boaMobileBanking', 4.4),
    ('Dashen Bank', 'Dashen Super App', 
     'com.dashen.dashensuperapp', 4.3),
]

cursor.executemany("""
    INSERT INTO banks (bank_name, app_name, play_store_id, avg_rating)
    VALUES (%s, %s, %s, %s)
""", banks_data)

print("✅ Banks inserted!")

# Verify
cursor.execute("SELECT * FROM banks;")
rows = cursor.fetchall()
for row in rows:
    print(f"  {row}")

In [ ]:
# Load the processed reviews
df = pd.read_csv('../data/raw/reviews_with_sentiment.csv')
print(f"✅ Loaded {len(df)} reviews")
print(df.columns.tolist())

# Get bank_id mapping
cursor.execute("SELECT bank_id, bank_name FROM banks;")
bank_map = {name: bid for bid, name in cursor.fetchall()}
print(f"\nBank ID mapping: {bank_map}")

# Map bank names to IDs
df['bank_id'] = df['bank'].map(bank_map)

# Check for unmapped banks
print(f"\nUnmapped banks: {df[df['bank_id'].isna()]['bank'].unique()}")

In [ ]:
# Insert reviews in batches
print("Inserting reviews into database...")

inserted = 0
errors = 0

for _, row in df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO reviews 
            (bank_id, review_text, rating, review_date, 
             sentiment_label, sentiment_score, identified_theme, source)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            int(row['bank_id']),
            str(row['review_text']),
            int(row['rating']),
            str(row['date']),
            str(row['sentiment_label']),
            float(row['sentiment_score']),
            str(row['identified_theme']),
            str(row['source'])
        ))
        inserted += 1
    except Exception as e:
        errors += 1

print(f"✅ Inserted: {inserted} reviews")
print(f"❌ Errors: {errors}")

In [ ]:
# Verification queries
print("=== DATA INTEGRITY VERIFICATION ===\n")

# Count per bank
cursor.execute("""
    SELECT b.bank_name, COUNT(r.review_id) as review_count,
           ROUND(AVG(r.rating)::numeric, 2) as avg_rating
    FROM banks b
    LEFT JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY review_count DESC;
""")
print("Reviews per bank:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} reviews | Avg rating: {row[2]}")

# Check nulls
cursor.execute("""
    SELECT 
        COUNT(*) FILTER (WHERE review_text IS NULL) as null_text,
        COUNT(*) FILTER (WHERE rating IS NULL) as null_rating,
        COUNT(*) FILTER (WHERE sentiment_label IS NULL) as null_sentiment
    FROM reviews;
""")
nulls = cursor.fetchone()
print(f"\nNull values — text: {nulls[0]}, rating: {nulls[1]}, sentiment: {nulls[2]}")

# Total count
cursor.execute("SELECT COUNT(*) FROM reviews;")
total = cursor.fetchone()[0]
print(f"\nTotal reviews in database: {total}")
print("✅ Verification complete!")

In [ ]:
# Save schema to SQL file
schema_sql = """
-- bank_reviews Database Schema
-- 10 Academy KAIM Week 2 — Fintech Review Analytics
-- Author: Samrawit Alemu

CREATE TABLE banks (
    bank_id SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL,
    app_name VARCHAR(100) NOT NULL,
    play_store_id VARCHAR(100),
    avg_rating FLOAT
);

CREATE TABLE reviews (
    review_id SERIAL PRIMARY KEY,
    bank_id INTEGER REFERENCES banks(bank_id),
    review_text TEXT,
    rating INTEGER CHECK (rating BETWEEN 1 AND 5),
    review_date DATE,
    sentiment_label VARCHAR(20),
    sentiment_score FLOAT,
    identified_theme VARCHAR(100),
    source VARCHAR(50) DEFAULT 'Google Play'
);
"""

with open('../scripts/schema.sql', 'w') as f:
    f.write(schema_sql)

print("✅ Schema saved to scripts/schema.sql")

# Close connection
cursor.close()
conn.close()
print("✅ Database connection closed!")